#### Task 05  ·  Full ETL System

In [8]:
import pandas as pd
import requests
import mysql.connector
from datetime import datetime

# Logging function
def log(message):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f"[{timestamp}] {message}")

In [9]:
# EXTRACT: Fetch data from API with error handling
def extract():
    log("Starting EXTRACT phase...")
    try:
        url = "https://jsonplaceholder.typicode.com/posts"
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        
        data = response.json()
        log(f"Successfully fetched {len(data)} records from API")
        return data
    except requests.exceptions.RequestException as e:
        log(f"ERROR in EXTRACT: {e}")
        return None

In [10]:
# TRANSFORM: Clean and enrich data
def transform(data):
    log("Starting TRANSFORM phase...")
    
    df = pd.DataFrame(data)
    initial_rows = len(df)
    log(f"Initial rows: {initial_rows}")
    
    # Keep relevant columns
    df = df[['userId', 'id', 'title', 'body']]
    log("Kept columns: userId, id, title, body")
    
    # Clean: Remove duplicates
    df = df.drop_duplicates(subset=['id'])
    log(f"After removing duplicates: {len(df)} rows")
    
    # Clean: Drop nulls
    df = df.dropna()
    log(f"After dropping nulls: {len(df)} rows")
    
    # Clean: Convert userId to integer
    df['userId'] = pd.to_numeric(df['userId'], errors='coerce').astype('Int64')
    log("Converted userId to integer")
    
    # Transform: Add word_count column
    df['word_count'] = df['title'].str.split().str.len()
    log("Added word_count column")
    
    # Transform: Add body_length column
    df['body_length'] = df['body'].str.len()
    log("Added body_length column")
    
    # Clean: Lowercase and strip title
    df['title'] = df['title'].str.lower().str.strip()
    df['body'] = df['body'].str.strip()
    log("Cleaned text fields")
    
    final_rows = len(df)
    log(f"Transform complete: {final_rows} rows ({initial_rows - final_rows} rows removed)")
    
    return df

In [11]:
# LOAD: Save to CSV and MySQL
def load(df, run_number=1):
    log("Starting LOAD phase...")
    
    # Load to CSV
    csv_file = f'etl_data_run{run_number}.csv'
    df.to_csv(csv_file, index=False)
    log(f"Saved {len(df)} rows to {csv_file}")
    
    # Load to MySQL
    try:
        conn = mysql.connector.connect(
            host='localhost',
            user='root',
            password='',
            database='etl_db'
        )
        cursor = conn.cursor()
        
        # Create table if not exists
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS posts (
                id INT PRIMARY KEY,
                userId INT,
                title VARCHAR(255),
                body TEXT,
                word_count INT,
                body_length INT
            )
        ''')
        
        # For run 2, check for duplicates before inserting
        if run_number == 2:
            log("Run 2: Checking for duplicates before insert...")
            for index, row in df.iterrows():
                cursor.execute('SELECT id FROM posts WHERE id = %s', (row['id'],))
                if cursor.fetchone() is None:
                    cursor.execute('''
                        INSERT INTO posts (id, userId, title, body, word_count, body_length)
                        VALUES (%s, %s, %s, %s, %s, %s)
                    ''', (row['id'], row['userId'], row['title'], row['body'], row['word_count'], row['body_length']))
        else:
            # Run 1: Clear and insert all
            cursor.execute('DELETE FROM posts')
            for index, row in df.iterrows():
                cursor.execute('''
                    INSERT INTO posts (id, userId, title, body, word_count, body_length)
                    VALUES (%s, %s, %s, %s, %s, %s)
                ''', (row['id'], row['userId'], row['title'], row['body'], row['word_count'], row['body_length']))
        
        conn.commit()
        log(f"Loaded data to MySQL database")
        conn.close()
    except mysql.connector.Error as e:
        log(f"ERROR in MySQL LOAD: {e}")
    
    log("LOAD phase complete")

In [12]:
# MAIN PIPELINE - Run 1
log("\n" + "="*60)
log("PIPELINE RUN 1")
log("="*60)

data = extract()
if data:
    df = transform(data)
    load(df, run_number=1)
    log("Pipeline Run 1 completed successfully!")
else:
    log("Pipeline Run 1 failed: No data extracted")

[2026-05-11 22:57:08] 
[2026-05-11 22:57:08] PIPELINE RUN 1
[2026-05-11 22:57:08] ============================================================
[2026-05-11 22:57:08] Starting EXTRACT phase...
[2026-05-11 22:57:08] Successfully fetched 100 records from API
[2026-05-11 22:57:08] Starting TRANSFORM phase...
[2026-05-11 22:57:08] Initial rows: 100
[2026-05-11 22:57:08] Kept columns: userId, id, title, body
[2026-05-11 22:57:08] After removing duplicates: 100 rows
[2026-05-11 22:57:08] After dropping nulls: 100 rows
[2026-05-11 22:57:08] Converted userId to integer
[2026-05-11 22:57:08] Added word_count column
[2026-05-11 22:57:08] Added body_length column
[2026-05-11 22:57:08] Cleaned text fields
[2026-05-11 22:57:08] Transform complete: 100 rows (0 rows removed)
[2026-05-11 22:57:08] Starting LOAD phase...
[2026-05-11 22:57:08] Saved 100 rows to etl_data_run1.csv
[2026-05-11 22:57:08] ERROR in MySQL LOAD: 1045 (28000): Access denied for user 'root'@'localhost' (using password: NO)
[2026-05

In [13]:
# MAIN PIPELINE - Run 2 (with duplicate prevention)
log("\n" + "="*60)
log("PIPELINE RUN 2 (Duplicate Prevention)")
log("="*60)

data = extract()
if data:
    df = transform(data)
    load(df, run_number=2)
    log("Pipeline Run 2 completed successfully!")
    log("Note: Duplicate rows were NOT inserted into the database")
else:
    log("Pipeline Run 2 failed: No data extracted")

[2026-05-11 22:57:08] 
[2026-05-11 22:57:08] PIPELINE RUN 2 (Duplicate Prevention)
[2026-05-11 22:57:08] ============================================================
[2026-05-11 22:57:08] Starting EXTRACT phase...
[2026-05-11 22:57:08] Successfully fetched 100 records from API
[2026-05-11 22:57:08] Starting TRANSFORM phase...
[2026-05-11 22:57:08] Initial rows: 100
[2026-05-11 22:57:08] Kept columns: userId, id, title, body
[2026-05-11 22:57:08] After removing duplicates: 100 rows
[2026-05-11 22:57:08] After dropping nulls: 100 rows
[2026-05-11 22:57:08] Converted userId to integer
[2026-05-11 22:57:08] Added word_count column
[2026-05-11 22:57:08] Added body_length column
[2026-05-11 22:57:08] Cleaned text fields
[2026-05-11 22:57:08] Transform complete: 100 rows (0 rows removed)
[2026-05-11 22:57:08] Starting LOAD phase...
[2026-05-11 22:57:08] Saved 100 rows to etl_data_run2.csv
[2026-05-11 22:57:08] ERROR in MySQL LOAD: 1045 (28000): Access denied for user 'root'@'localhost' (using

In [21]:
# Summary statistics
log("\n" + "—"*80)
log("FINAL SUMMARY")
log("—"*60)
print(f"\nFinal DataFrame shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nTop 5 records:")
print(df[['id', 'userId', 'title', 'word_count', 'body_length']].head())
print(f"\nData types:\n{df.dtypes}")

[2026-05-11 23:05:24] 
————————————————————————————————————————————————————————————————————————————————
[2026-05-11 23:05:24] FINAL SUMMARY
[2026-05-11 23:05:24] ————————————————————————————————————————————————————————————

Final DataFrame shape: (100, 6)

Columns: ['userId', 'id', 'title', 'body', 'word_count', 'body_length']

Top 5 records:
   id  userId                                              title  word_count  \
0   1       1  sunt aut facere repellat provident occaecati e...           9   
1   2       1                                       qui est esse           3   
2   3       1  ea molestias quasi exercitationem repellat qui...           9   
3   4       1                               eum et est occaecati           4   
4   5       1                                 nesciunt quas odio           3   

   body_length  
0          158  
1          206  
2          164  
3          190  
4          147  

Data types:
userId         Int64
id             int64
title            